In [ ]:
'''
In our original WF model, we assumed both MIC and MAC have single-locus sized chromosomes. 

Here based on our original WF model, we further introduced multiple loci within MIC chromosomes as MIC chromosomes are much less
fragmented than MAC ones. We wanted to investigate the effect of MIC chromosome fragmentation on eliminating deleterious mutations.

Here we assumed no crossover events during generation of gametes. Once zygotes developing into new MAC, the chromosome will first
be fragmented and then amplified to 45-ploid.

'''

In [ ]:
from __future__ import division
import numpy as np
from scipy import stats
import pandas as pd
import matplotlib.pyplot as plt
import time
import pickle

In [ ]:
class Populations(object):

    def __init__(self,nReps,nInds,nLoci,num_loci, ploidy=45,genomic_mu=0.1,selcoef=0.1, model_version ='ADD'):

        self.nReps = nReps
        self.nInds = nInds
        self.nLoci = nLoci
        self.num_loci = num_loci # How many loci in each germline chromosome
        
        self.soma = np.zeros((nReps,nInds,nLoci),dtype='int')
        
        self.construct_germ()
        
        self.ploidy = ploidy
        self.mu = genomic_mu/(nLoci*ploidy)
        self.selcoef = selcoef
       
        self.current_step = 0

        self.model_version = model_version
        

        
    def construct_germ(self):
        
        germ_chromosome = np.zeros((self.num_loci), dtype ='int')
        germ_each_chromosome = np.repeat(germ_chromosome[np.newaxis], 2, axis =0)
        self.num_chrom = int(self.nLoci/self.num_loci)  # How many set of chromosomes in Germ
        
        ind_germ = np.repeat(germ_each_chromosome[np.newaxis], self.num_chrom, axis =0)
        pop_germ = np.repeat(ind_germ[np.newaxis], self.nInds, axis =0)
        
        self.germ = np.repeat(pop_germ[np.newaxis], self.nReps, axis =0)
        
        
        
    def fitness(self):
        """return a numpy array containing the fitness for each individual in each population"""
        ws = (self.soma.astype('float')/self.ploidy)*self.selcoef

        if self. model_version == 'ADD':
            fitnesses = 1 - np.sum(ws,axis=2)
            
        elif self. model_version == 'MUL':
            each_locus = 1-ws
            fitnesses = np.prod(each_locus, axis =2)
            
        fitnesses[fitnesses <= 0] = 0  

        return fitnesses

            
    def relative_fitness(self):
        """return a numpy array containing each individual's relative fitness"""
        w = self.fitness()
        total_w = np.sum(w,axis=1)
        totalw = np.expand_dims(total_w,axis=1)
        relfit = w/totalw
        return relfit

                
        
        
    #Wright_Fisher model
    def mutate_all_before(self):
        """Mutate all individuals in the population with mutation rate mu.
        Wright-Fisher model method. Use this one instead of mutate_all if you want to mutate the population before selection."""
        wt_soma = self.ploidy - self.soma
        wt_germ = 1 - self.germ
        soma_mutations = np.random.binomial(wt_soma,self.mu)
        germ_mutations = np.random.binomial(wt_germ,self.mu)
        self.soma += soma_mutations
        self.germ += germ_mutations
        return
    
    
    
    def pick_parents_all(self):
        """Randomly choose N parents to produce offspring to populate the next generation.
        Each individual's probability of being chosen is weighted by its relative fitness.
        Wright-Fisher model method."""
        nReps,nInds = self.soma.shape[0:2]        
        relfit = self.relative_fitness()
        csrelfit = np.cumsum(relfit,axis=1)
        randvals = np.random.random((nReps,nInds))
        parents = map(np.searchsorted,csrelfit,randvals)
        return parents
    
    
    
    def mitosis_all(self,parents):
        """Generate mitotic offspring from all individuals selected as parents.
        Wright-Fisher model method."""
        nReps,nInds = self.soma.shape[0:2]        
        rReps = np.ones((nReps,nInds),dtype='int')*np.expand_dims(np.arange(nReps),axis=1)
        self.soma = self.soma[rReps,parents,]
        self.germ = self.germ[rReps,parents,]
        return
    
    
    def amitosis_all(self,parents):
        """Generate amitotic offspring from all individuals selected as parents. Only one
        amitotic offspring is generated from each parent, so this method does not reflect
        the reciprocity of amitosis.
        Wright-Fisher model method."""
        nReps,nInds = self.soma.shape[0:2]        
        rReps = np.ones((nReps,nInds),dtype='int')*np.expand_dims(np.arange(nReps),axis=1)
        good = (self.ploidy-self.soma[rReps,parents,])*2
        bad = self.soma[rReps,parents,]*2
        self.soma = np.random.hypergeometric(bad,good,self.ploidy)
        self.germ = self.germ[rReps,parents,]
        return
    
    
    def wright_fisher_step(self,asex_type='amitosis'):
        """Take populations through a single Wright-Fisher model generation"""
        self.mutate_all_before()
        parents = self.pick_parents_all()
        if asex_type == 'amitosis':
            self.amitosis_all(parents)
        elif asex_type == 'mitosis':
            self.mitosis_all(parents)
        self.current_step += 1
        return
    
    
    

    
    
    #Asexul reproduction_WF model and Moran model
    
    def step(self,model='M',asex_type='amitosis',sex_freq=None):
        """Take populations through one time step if model='M', or one generation if model='WF'"""
        if model == 'M':
            self.moran_step(asex_type)
        elif model == 'WF':
            self.wright_fisher_step(asex_type)
        return 
    


    def make_gametes(self, parents):
        rReps = np.ones((self.nReps,self.nInds),dtype='int')*np.expand_dims(np.arange(self.nReps),axis=1)
        germ = self.germ[rReps, parents, ]
        
        gametes = []
        picked_index = np.random.randint(0,2, size =((self.nReps, self.nInds, self.num_chrom)))
        
        for i in range(self.nReps):
            pop_gametes = []
            for j in range(self.nInds):
                ind_gametes = []
                for k in range(self.num_chrom):
                    ind_gametes.append([germ[i][j][k][picked_index[i][j][k]]])
                pop_gametes.append(ind_gametes)
            gametes.append(pop_gametes)
            
        gametes = np.array(gametes)
#         final_gametes = np.expand_dims(gametes, axis =3)

        return gametes

   
    def make_zygotes(self, random_mating=False):

        self.mutate_all_before()

        if random_mating == False:
            parents = self.pick_parents_all()

            these_gametes = self.make_gametes(parents)
            those_gametes = self.make_gametes(parents)

        else: 
            first_parents = self.pick_parents_all()
            second_parents = self.pick_parents_all()

            these_gametes = self.make_gametes(first_parents)
            those_gametes = self.make_gametes(second_parents)           

        
        zygotes = np.append(these_gametes, those_gametes, axis =3)  

        return zygotes

    
    def get_results(self):
        """calculate stuff like mean fitness, Gst, and number of fixed mutations"""

        
        W = self.fitness()

        mW = np.nanmean(W)
        log_mW = np.log(mW)


        pop_mean_fit = np.nanmean(W, axis =1)  # Calculate mean fitness from another way: first get the mean fitness of each population
        mean_fit = np.nanmean(pop_mean_fit)  # Then average across the populations

        pop_mfit_std = np.nanstd(pop_mean_fit)    # Get the STD of population mean fitness across each population
        pop_mfit_lower = np.percentile(pop_mean_fit, 2.5)   # The 2.5% lower bound of population mean fitness among populations
        pop_mfit_upper = np.percentile(pop_mean_fit, 97.5)   # The 97.5% upper bound of population mean fitness among populations


        total_fit_var = []     
        for i in range(self.nReps):  
            fit_var = np.var(W[i])   # Variance within each population
            total_fit_var.append(fit_var)


        varW_mean = np.nanmean(total_fit_var)  # Mean variance among populations

        varW_std = np.nanstd(total_fit_var)    # STD of variance among populations
        varW_lower = np.percentile(total_fit_var, 2.5)   # 2.5% lower bound of variance among populations
        varW_upper = np.percentile(total_fit_var, 97.5)  # 97.5% upper bound of variance among populations


        pop_fit_lower = np.percentile(W, 25, axis =1)   # The 25% lower bound of fitness within each population
        pop_fit_upper = np.percentile(W, 75, axis =1)   # The 75% upper bound of fitness within each population

        pop_fit_50_range = pop_fit_upper - pop_fit_lower  # The middle 50% fitness range within each population


        pop_fit_r50_mean = np.nanmean(pop_fit_50_range)   # Mean of middle 50% fitness range among populations
        pop_fit_r50_std = np.nanstd(pop_fit_50_range)    # STD of middle 50% fitness range among populations
        pop_fit_r50_lower = np.percentile(pop_fit_50_range, 2.5)  # 2.5% lower bound of middle 50% fitness range among populations
        pop_fit_r50_upper = np.percentile(pop_fit_50_range, 97.5) # 97.5% lower bound of middle 50% fitness range among populations

        
            
        return [mW, log_mW, mean_fit, pop_mfit_std, pop_mfit_lower, pop_mfit_upper, varW_mean, varW_std, varW_lower, varW_upper, \
        pop_fit_r50_mean, pop_fit_r50_std, pop_fit_r50_lower, pop_fit_r50_upper]


    

    def sex(self,random_mating=False):
        
        zygotes = self.make_zygotes(random_mating)  # make zygotes. Here the random_mating is just an argument of self.make
                                                    # _zygotes, doesn't mean that here is the random_mating
        self.germ = zygotes  # the germline genome will be the same as the zygotes.
        
        zy_sum = np.sum(zygotes, axis =3)
        final_zygotes = zy_sum.reshape(self.nReps, self.nInds, self.nLoci)
        
        hetcount = len(final_zygotes[final_zygotes==1])
        
        hetvals = int(self.ploidy/2) + np.random.binomial(self.ploidy-2*int(self.ploidy/2),0.5, hetcount)
        
        self.soma[final_zygotes==1] = hetvals
        self.soma[final_zygotes==0] = 0
        self.soma[final_zygotes==2] = self.ploidy
        
        self.current_step +=1
                             
        return 

        
    def simulate(self,stepcount,model='M',asex_type='amitosis',sex_freq=None,random_mating=False,strides=10):

        results = [self.get_results()]
        

        start = time.time()
    
        while self.current_step <= stepcount:
            if (sex_freq != None) and (self.current_step%sex_freq == 0):
                self.sex(random_mating)
                
            else:
                self.step(model,asex_type,sex_freq)

            
            if self.current_step%strides == 0:
                results.append(self.get_results())


        colnames = ['meanFit','ln_meanFit','PopMeanFit_Mean', 'PopMeanFit_STD', 'PopMeanFit_Lower', 'PopMeanFit_Upper', \
        'PopVar_Mean', 'PopVar_STD', 'PopVar_Lower', 'PopVar_Upper', 'PopR50Fit_Mean', 'PopR50Fit_STD', 'PopR50Fit_Lower', \
        'PopR50Fit_Upper']


        results = pd.DataFrame(np.array(results),columns=colnames)
        


        end = time.time()
        print 'TOTAL TIME: ',end-start
        return results
        

    def simulateNsave(self,outfile_1,stepcount,model='M',asex_type='amitosis',sex_freq=None,random_mating=False,strides=10):
        """same as simulate method except results are written to the file specified by outfile"""
        results = self.simulate(stepcount,model,asex_type,sex_freq,random_mating,strides)
        
        result = results
        result.to_csv(outfile_1,index=False)
        
        return
         


        
def save_object(obj, filename):
    with open(filename, 'wb') as output:
        pickle.dump(obj, output, pickle.HIGHEST_PROTOCOL)

In [ ]:
a =Populations(nReps =500, nInds =20, nLoci =100, num_loci =5, ploidy =45, genomic_mu=0.1, selcoef =0.1, model_version ='MUL')
a.simulateNsave('Fitness_WF_RM_20_E1_GermL_5PerCh_FirstFagThenAmp.csv', 10*1000, model ='WF', asex_type='amitosis',sex_freq=1,random_mating=True,strides=1)


b =Populations(nReps =500, nInds =20, nLoci =100, num_loci =5, ploidy =45, genomic_mu=0.1, selcoef =0.1, model_version ='MUL')
b.simulateNsave('Fitness_WF_SF_20_E1_GermL_5PerCh_FirstFagThenAmp.csv', 10*1000, model ='WF', asex_type='amitosis',sex_freq=1,random_mating=False,strides=1)